In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_ext_lpfxb364_extfile(
    spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ExtLPFXB364Id string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "ExtLPFXB364Key string,"
        #     "RestructuringDate string,"
        #     "NewExposureKey string,"
        #     "OldExposureKey string,"
        #     "RestructuringTypeCode string,"
        #     "RestructuringType string,"
        #     "RestructuringAmount string,"
        #     "TotalOutstandingAmount string,"
        #     "IndPortfolio string,"
        #     "ExposureType string,"
        #     "RestructuringProductKey string,"
        #     "AgreementKey string,"
        #     "CDI string,"
        #     "PartyTaxRegistrationNumber string,"
        #     "RestructuringChannel string,"
        #     "HasCollateralFlag string,"
        #     "Forbearance string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExtLPFXB364"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "extfile",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================= lpfxb364 ====================

        lpfxb364_extra_filters = None

        if lpfxb364_extra_filters is not None:
            lpfxb364_extra_filters.append(date_filter)
            lpfxb364_final_filters = " AND ".join(lpfxb364_extra_filters)
        else:
            lpfxb364_final_filters = date_filter

        lpfxb364_selected_fields = [
            "RestructuringDate",
            "OldExposureKey",
            "NewExposureKey",
            "AgreementKey",
            "CDI",
            "PartyTaxRegistrationNumber",
            "RestructuringTypeCode",
            "RestructuringType",
            "TotalOutstandingAmount",
            "RestructuringAmount",
            "IndPortfolio",
            "ExposureType",
            "RestructuringProductKey",
            "RestructuringChannel",
            "HasCollateralFlag",
            "Forbearance",
            "BusinessDate",
        ]

        lpfxb364 = load_landed_delta_table(
            spark,
            externalfiles_zone,  # type: ignore
            "extlpfxb364",
            lpfxb364_final_filters,
            lpfxb364_selected_fields,
        )

        # ================= lpfxb364_df =====================

        ExtLPFXB364Id_primary_key_fields = [
            lpfxb364.RestructuringDate,
            lpfxb364.OldExposureKey,
            lpfxb364.NewExposureKey,
            lit("extfile"),
        ]

        lpfxb364_df = lpfxb364.withColumn(
            "ExtLPFXB364Id",
            sha2(concat_ws(";", *ExtLPFXB364Id_primary_key_fields), 256),
        ).withColumn(
            "ExtLPFXB364Key", concat_ws(";", *ExtLPFXB364Id_primary_key_fields)
        )

        lpfxb364_result_df = lpfxb364_df.select(
            lpfxb364_df.ExtLPFXB364Id.alias("ExtLPFXB364Id"),
            lit("extfile").alias("SourceSystem"),
            lit("-9999").cast("string").alias("PortfolioKey"),
            lit("Core Retail").cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            lpfxb364_df.ExtLPFXB364Key.alias("ExtLPFXB364Key"),
            lpfxb364_df.RestructuringDate.alias("RestructuringDate"),
            lpfxb364.NewExposureKey.alias("NewExposureKey"),
            lpfxb364_df.OldExposureKey.alias("OldExposureKey"),
            lpfxb364_df.RestructuringTypeCode.alias("RestructuringTypeCode"),
            lpfxb364_df.RestructuringType.alias("RestructuringType"),
            lpfxb364_df.RestructuringAmount.alias("RestructuringAmount"),
            lpfxb364_df.TotalOutstandingAmount.alias("TotalOutstandingAmount"),
            lpfxb364_df.IndPortfolio.alias("IndPortfolio"),
            lpfxb364_df.ExposureType.alias("ExposureType"),
            lpfxb364_df.RestructuringProductKey.alias("RestructuringProductKey"),
            lpfxb364_df.AgreementKey.alias("AgreementKey"),
            lpfxb364_df.CDI.alias("CDI"),
            lpfxb364_df.PartyTaxRegistrationNumber.alias("PartyTaxRegistrationNumber"),
            lpfxb364_df.RestructuringChannel.alias("RestructuringChannel"),
            lpfxb364_df.HasCollateralFlag.alias("HasCollateralFlag"),
            lpfxb364_df.Forbearance.alias("Forbearance"),
            lpfxb364_df.BusinessDate.alias("BusinessDate"),
        )

        # ================== Append ==================

        lpfxb364_result_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                externalfiles_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_ext_lpfxb364_extfile(spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix)